In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('spark://spark-master:7077')
    .config('spark.executor.instances', 1)  # Is ignored by spark standalone cluster manager
    .config('spark.cores.max', '4')         # Workaround to cap number of executor's instances
    .config('spark.executor.cores', 2)
    .config('spark.executor.memory', '512M')
    .getOrCreate()
)

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 00:54:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
emp_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp_df = spark.read.format('csv').schema(emp_schema).option('header', True).load('/home/spark/data/employee/1')

In [8]:
emp_df.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)900-5337|604738.0|            8|
|    Ashley|   Montoya|        C

In [22]:
dept_schema = "department_id int, department_name string, description string, city string, state string, country string"

dept_df = spark.read.format('csv').schema(dept_schema).option('header', True).load('/home/spark/data/department/1')

In [23]:
dept_df.show()

+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|department_id|     department_name|         description|                city|state|            country|
+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|            1|         Bryan-James|Optimized disinte...|        Melissaburgh|   FM|Trinidad and Tobago|
|            2|Smith, Craig and ...|Digitized empower...|          Morrisside|   DE|          Sri Lanka|
|            3|Pittman, Hess and...|Multi-channeled c...|         North David|   SC|       Turkmenistan|
|            4|Smith, Snyder and...|Reactive neutral ...|       Lake Jennifer|   TX|         Madagascar|
|            5|          Hardin Inc|Re-contextualized...|           Hayestown|   WA|               Fiji|
|            6|         Sanders LLC|Innovative multim...|         Phamchester|   TN|         Micronesia|
|            7|         Ward-Gordon|Progressive logis..

In [24]:
emp_joined_df = emp_df.join(dept_df, how='inner', on='department_id')

In [25]:
emp_joined_df.explain()

== Physical Plan ==
*(5) Project [department_id#49, first_name#42, last_name#43, job_title#44, dob#45, email#46, phone#47, salary#48, department_name#238, description#239, city#240, state#241, country#242]
+- *(5) SortMergeJoin [department_id#49], [department_id#237], Inner
   :- *(2) Sort [department_id#49 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(department_id#49, 200), ENSURE_REQUIREMENTS, [plan_id=308]
   :     +- *(1) Filter isnotnull(department_id#49)
   :        +- FileScan csv [first_name#42,last_name#43,job_title#44,dob#45,email#46,phone#47,salary#48,department_id#49] Batched: false, DataFilters: [isnotnull(department_id#49)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/spark/data/employee/1], PartitionFilters: [], PushedFilters: [IsNotNull(department_id)], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
   +- *(4) Sort [department_id#237 ASC NULLS FIRST], false, 0
      +- Ex

In [26]:
from pyspark.sql.functions import spark_partition_id, col, count

emp_joined_df.withColumn('partition_id', spark_partition_id()).groupBy(col('partition_id')).agg(count(col('partition_id')).alias('partition_count')).show()

+------------+---------------+
|partition_id|partition_count|
+------------+---------------+
|         103|         100417|
|         122|          99780|
|          43|          99451|
|         107|          99805|
|          49|          99706|
|          51|         100248|
|         102|         100214|
|          66|         100210|
|         174|         100155|
|          89|         100014|
+------------+---------------+



In [28]:
emp_joined_df.groupBy('department_id').agg(count('department_id')).show()

+-------------+--------------------+
|department_id|count(department_id)|
+-------------+--------------------+
|            1|               99451|
|            6|               99706|
|            3|              100248|
|            5|              100210|
|            9|              100014|
|            4|              100214|
|            8|              100417|
|            7|               99805|
|           10|               99780|
|            2|              100155|
+-------------+--------------------+

